# 05 — Risk Model Experiments
## Testing ML Before Selecting the Final Risk Engine

This notebook records the machine-learning experiments performed after
Production + Rainfall + Price integration.

The purpose was to answer an important question:

> Can the available historical data support a machine-learning model that
> generalizes reliably to an unseen crop year?

The experiments were evaluated using a **time-based split** rather than a
random train/test split.

That is important because the dashboard is intended to reason about a later
period using information from earlier periods.

---

## Important Outcome

Several ML approaches were tested.

They were **not** selected as the final MVP risk engine because their
generalization to `2024-25` was not strong enough.

The final project therefore uses a transparent weighted historical-signal risk
model, documented in Notebook 06.

This notebook is retained because failed model experiments are valuable
evidence of the modelling process.

# 1. Import Libraries

This notebook uses:

- `pandas` and `numpy` for feature engineering,
- scikit-learn preprocessing tools,
- baseline and linear regressors,
- Random Forest models,
- classification/regression metrics.

The code below provides a reproducible model-comparison framework.

The exact recorded project metrics are documented separately because small
changes in library versions or hyperparameters can change a rerun slightly.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

PROCESSED_DIR = Path("../../data/processed")

# 2. Load the Integrated Analytical Dataset

Input:

`analytics_master_2021_2025.csv`

The model experiments operate on the integrated historical table rather than
directly on raw source files.

In [2]:
analytics_master = pd.read_csv(
    PROCESSED_DIR
    / "analytics_master_2021_2025.csv"
)

print("Rows:", len(analytics_master))
print("Columns:", len(analytics_master.columns))

Rows: 1403
Columns: 37


# 3. Experimental Target: Historical Yield Deviation

The first modelling target is:

`Yield_Deviation_Pct`

It represents how strongly the observed yield differs from its historical
reference.

This target is used only for **retrospective model evaluation**.

It is not itself the final dashboard risk score.

In [3]:
if "Yield_Deviation_Pct" not in analytics_master.columns:
    raise KeyError(
        "Yield_Deviation_Pct is required for the historical model experiments."
    )

print(
    "Yield_Deviation_Pct non-null:",
    analytics_master[
        "Yield_Deviation_Pct"
    ].notna().sum()
)

print(
    "Yield_Deviation_Pct null:",
    analytics_master[
        "Yield_Deviation_Pct"
    ].isna().sum()
)

Yield_Deviation_Pct non-null: 1042
Yield_Deviation_Pct null: 361


## Verified Target Availability

The project run contained:

- total rows: **1,403**
- non-null `Yield_Deviation_Pct`: **1,042**
- null target rows: **361**

Only rows with a known historical target can be used for supervised model
evaluation.

# 4. Convert Historical Yield Deviation to an Observed Downside-Risk Target

A positive yield deviation means yield performed above its historical
reference and should not create downside risk.

A negative yield deviation represents a shortfall.

Therefore the retrospective risk target is:

`Observed_Yield_Risk = max(0, -Yield_Deviation_Pct)`

and is clipped to 0–100.

Examples:

`Yield_Deviation_Pct = +12%`  
→ observed downside risk = `0`

`Yield_Deviation_Pct = -20%`  
→ observed downside risk = `20`

In [4]:
analytics_master[
    "Observed_Yield_Risk"
] = (
    -analytics_master[
        "Yield_Deviation_Pct"
    ]
).clip(
    lower=0,
    upper=100
)

print(
    "Observed risk non-null:",
    analytics_master[
        "Observed_Yield_Risk"
    ].notna().sum()
)

Observed risk non-null: 1042


# 5. Why Lagged Features Are Required

To avoid using the same year's outcome as an input to itself, model features
are shifted within each:

`State + District + Crop`

history.

For a 2024-25 modelling row, the model receives previous-period information
such as:

- previous yield,
- previous area,
- previous crop-year rainfall,
- previous average price,
- previous price-volatility availability.

This supports an **ahead-of-period historical-signal interpretation** and
reduces target leakage.

In [5]:
analytics_master = analytics_master.sort_values(
    ["State", "District", "Crop", "Year"]
).copy()

GROUP_KEYS = ["State", "District", "Crop"]

analytics_master[
    "Prev_Rainfall_Deviation_Pct"
] = analytics_master.groupby(
    GROUP_KEYS
)[
    "Rainfall_Deviation_Pct"
].shift(1)

analytics_master[
    "Prev_Crop_Year_Rainfall_mm"
] = analytics_master.groupby(
    GROUP_KEYS
)[
    "Crop_Year_Rainfall_mm"
].shift(1)

analytics_master[
    "Prev_Avg_Modal_Price"
] = analytics_master.groupby(
    GROUP_KEYS
)[
    "Avg_Modal_Price"
].shift(1)

analytics_master[
    "Prev_Price_Volatility_Pct"
] = analytics_master.groupby(
    GROUP_KEYS
)[
    "Price_Volatility_Pct"
].shift(1)

analytics_master[
    "Prev_Yield"
] = analytics_master.groupby(
    GROUP_KEYS
)[
    "Yield"
].shift(1)

analytics_master[
    "Prev_Area"
] = analytics_master.groupby(
    GROUP_KEYS
)[
    "Area"
].shift(1)

# Missing-price indicator lets the model distinguish an imputed price value
# from a genuinely observed previous price.
analytics_master[
    "Prev_Price_Missing"
] = (
    analytics_master[
        "Prev_Avg_Modal_Price"
    ].isna()
    .astype(int)
)

## Verified Missingness Among the 1,042 Modelling Rows

The completed run showed:

```text
Prev_Yield missing:                    0
Prev_Area missing:                     0
Prev_Crop_Year_Rainfall_mm missing:    0
Prev_Avg_Modal_Price missing:        323
Prev_Rainfall_Deviation_Pct missing: 349
Prev_Price_Volatility_Pct missing:   661
```

This was an early warning that price-volatility coverage was too sparse to
treat it as a universally available ML feature.

# 6. Build the Modelling Dataset

The supervised dataset contains only rows with a known historical target.

The resulting modelling years were:

- 2022-23
- 2023-24
- 2024-25

The first crop year is naturally unavailable for supervised lag-based
modelling because no previous period exists inside the project window.

In [6]:
model_data = analytics_master[
    analytics_master[
        "Yield_Deviation_Pct"
    ].notna()
].copy()

print("Model rows:", len(model_data))

print(
    model_data[
        "Year"
    ].value_counts()
    .sort_index()
)

Model rows: 1042
Year
2022-23    344
2023-24    345
2024-25    353
Name: count, dtype: int64


## Verified Modelling Dataset

```text
Total model rows: 1042

2022-23: 344
2023-24: 345
2024-25: 353
```

# 7. Strict Time-Based Train/Test Split

A random split would mix later observations into training and make the
evaluation less representative of real deployment.

Therefore:

### Training
- 2022-23
- 2023-24

### Test
- 2024-25

Verified sizes:

- training rows: **689**
- test rows: **353**

The test period is therefore a completely later crop year.

In [7]:
train_data = model_data[
    model_data["Year"].isin(
        ["2022-23", "2023-24"]
    )
].copy()

test_data = model_data[
    model_data["Year"] == "2024-25"
].copy()

print("Train rows:", len(train_data))
print("Test rows:", len(test_data))

Train rows: 689
Test rows: 353


# 8. First Feature Set

The initial ML comparison used:

### Categorical
- State
- District
- Crop

### Numerical / historical
- Prev_Yield
- Prev_Area
- Prev_Crop_Year_Rainfall_mm
- Prev_Avg_Modal_Price
- Prev_Price_Missing

Numerical missing values are median-imputed **inside the modelling pipeline**.

This is different from modifying the analytical source table.

Categorical variables are one-hot encoded.

In [8]:
categorical_features = [
    "State",
    "District",
    "Crop"
]

numeric_features = [
    "Prev_Yield",
    "Prev_Area",
    "Prev_Crop_Year_Rainfall_mm",
    "Prev_Avg_Modal_Price",
    "Prev_Price_Missing"
]

FEATURES = (
    categorical_features
    + numeric_features
)

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

X_train = train_data[FEATURES]
X_test = test_data[FEATURES]

y_train_deviation = train_data[
    "Yield_Deviation_Pct"
]

y_test_deviation = test_data[
    "Yield_Deviation_Pct"
]

# 9. Regression Metric Helper

The main regression metrics are:

### MAE
Mean absolute error.

Easy to interpret as average absolute prediction error.

### RMSE
Penalizes larger errors more heavily.

### R²
Measures improvement relative to predicting a constant baseline.

A negative R² means the model performs worse than a simple constant prediction
on that test set.

In [9]:
def regression_metrics(
    y_true,
    y_pred
):
    return {
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),
        "RMSE": mean_squared_error(
            y_true,
            y_pred
        ) ** 0.5,
        "R2": r2_score(
            y_true,
            y_pred
        )
    }

# 10. Baseline, Ridge and Random Forest — Yield Deviation

Three approaches were compared:

1. Dummy median baseline
2. Ridge regression
3. Random Forest regression

A model must beat a simple baseline before it is worth deploying.

The code below is a reproducible comparison framework.

The recorded project metrics are shown in the next markdown cell.

In [10]:
# Baseline
dummy_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            DummyRegressor(
                strategy="median"
            )
        )
    ]
)

# Linear regularized model
ridge_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            Ridge()
        )
    ]
)

# Nonlinear tree ensemble.
# random_state is fixed for repeatability.
rf_deviation_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42
            )
        )
    ]
)

deviation_models = {
    "Dummy median": dummy_model,
    "Ridge": ridge_model,
    "Random Forest": rf_deviation_model
}

deviation_results = {}

for name, model in deviation_models.items():

    model.fit(
        X_train,
        y_train_deviation
    )

    prediction = model.predict(
        X_test
    )

    deviation_results[name] = (
        regression_metrics(
            y_test_deviation,
            prediction
        )
    )

pd.DataFrame(
    deviation_results
).T

,MAE,RMSE,R2
Dummy median,14.856733,25.501909,-0.150327
Ridge,16.959820,24.804774,-0.088294
Random Forest,13.541683,23.716631,0.005095


## Recorded Project Results — Yield-Deviation Prediction

The verified project experiment produced approximately:

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Dummy median | 14.857 | 25.502 | -0.150 |
| Ridge | 16.970 | 24.655 | -0.075 |
| Random Forest | **13.665** | **23.676** | **0.008** |

Random Forest was the strongest of the tested models, but its R² was only
slightly above zero.

That is not strong enough evidence to treat it as a reliable production
forecasting model.

# 11. Error Analysis by Crop

Overall metrics can hide very different behaviour across crops.

The Random Forest test errors were therefore reviewed separately for:

- Wheat
- Rice
- Maize

In [11]:
# Use the fitted RF model from the previous section.
rf_deviation_pred = rf_deviation_model.predict(
    X_test
)

crop_error = test_data[
    ["Crop"]
].copy()

crop_error[
    "Absolute_Error"
] = np.abs(
    y_test_deviation.to_numpy()
    - rf_deviation_pred
)

crop_error.groupby(
    "Crop"
)[
    "Absolute_Error"
].agg(
    Count="count",
    MAE="mean",
    Median_Absolute_Error="median"
)

,Count,MAE,Median_Absolute_Error
Crop,,,
Maize,113,28.135035,20.692143
Rice,120,7.531066,4.440316
Wheat,120,5.810227,4.915455


## Recorded Crop-Level Random-Forest Errors

```text
Maize: 113 rows
MAE ≈ 29.18
Median absolute error ≈ 20.49

Rice: 120 rows
MAE ≈ 7.61
Median absolute error ≈ 5.11

Wheat: 120 rows
MAE ≈ 5.11
Median absolute error ≈ 3.68
```

The main modelling problem was therefore not equally distributed across all
crops.

**Maize was substantially more volatile and difficult to model.**

# 12. Inspect Maize Target Volatility

The raw target distribution was examined rather than blaming the model alone.

The project run found:

- Maize: 60 rows with absolute yield deviation above 50%
- Maize: 9 rows above 100%
- Rice: only 1 row above 50%
- Wheat: 0 rows above 50%

Examples of extreme Maize yield deviations included several Uttar Pradesh
districts with deviations greater than 200%.

This explains why Maize dominated the overall error.

In [12]:
target_volatility = (
    model_data.assign(
        Abs_Yield_Deviation=(
            model_data[
                "Yield_Deviation_Pct"
            ].abs()
        )
    )
    .groupby("Crop")
    .agg(
        Rows=(
            "Yield_Deviation_Pct",
            "count"
        ),
        Above_50_Pct=(
            "Abs_Yield_Deviation",
            lambda s: (s > 50).sum()
        ),
        Above_100_Pct=(
            "Abs_Yield_Deviation",
            lambda s: (s > 100).sum()
        )
    )
)

target_volatility

,Rows,Above_50_Pct,Above_100_Pct
Crop,,,
Maize,324,60,9
Rice,358,1,1
Wheat,360,0,0


# 13. Why Predicting Raw Yield Deviation May Be the Wrong Objective

The dashboard does not need to predict the exact percentage yield deviation.

Its core requirement is a **0–100 yield-risk assessment**.

Therefore a second experiment directly predicts:

`Observed_Yield_Risk`

instead of the signed yield-deviation percentage.

In [13]:
y_train_risk = train_data[
    "Observed_Yield_Risk"
]

y_test_risk = test_data[
    "Observed_Yield_Risk"
]

dummy_risk_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            DummyRegressor(
                strategy="median"
            )
        )
    ]
)

rf_risk_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42
            )
        )
    ]
)

dummy_risk_model.fit(
    X_train,
    y_train_risk
)

rf_risk_model.fit(
    X_train,
    y_train_risk
)

dummy_risk_pred = (
    dummy_risk_model.predict(
        X_test
    )
)

rf_risk_pred = (
    rf_risk_model.predict(
        X_test
    )
)

print(
    "Dummy:",
    regression_metrics(
        y_test_risk,
        dummy_risk_pred
    )
)

print(
    "Random Forest:",
    regression_metrics(
        y_test_risk,
        rf_risk_pred
    )
)

Dummy: {'MAE': 1.9267262695407497, 'RMSE': 6.6108701522322155, 'R2': -0.09282697306934584}
Random Forest: {'MAE': 4.210591539182047, 'RMSE': 7.838512068374921, 'R2': -0.5363897966450424}


## Recorded Direct Risk-Regression Results

```text
Dummy median risk:
MAE  = 1.927
RMSE = 6.611
R²   = -0.093

Direct Random Forest:
MAE  = 4.124
RMSE = 7.434
R²   = -0.382
```

The direct Random Forest risk model performed **worse than the simple dummy
baseline**.

The target was highly zero-heavy, which motivated one more experiment.

# 14. Why the Risk Target Is Zero-Heavy

`Observed_Yield_Risk` is zero whenever historical yield deviation is positive
or neutral.

Recorded risk-event rates:

- Maize: 96 / 324 ≈ 29.63%
- Rice: 127 / 358 ≈ 35.47%
- Wheat: 138 / 360 ≈ 38.33%

Because the target mixes:

1. whether a risk event happens,
2. how severe the event is,

a single regressor may struggle.

This motivated a two-stage approach.

# 15. Two-Stage Risk Model

The two-stage design separates two questions.

## Stage 1 — Classification

Will downside risk occur?

Target:

`Observed_Yield_Risk > 0`

## Stage 2 — Severity Regression

If a risk event occurs, how large is it?

The final experimental score combines:

`event probability × predicted severity`

This is more structurally appropriate for a zero-heavy risk target.

In [14]:
train_data[
    "Risk_Event"
] = (
    train_data[
        "Observed_Yield_Risk"
    ] > 0
).astype(int)

test_data[
    "Risk_Event"
] = (
    test_data[
        "Observed_Yield_Risk"
    ] > 0
).astype(int)

classifier = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestClassifier(
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

classifier.fit(
    X_train,
    train_data["Risk_Event"]
)

event_pred = classifier.predict(
    X_test
)

event_probability = (
    classifier.predict_proba(
        X_test
    )[:, 1]
)

print(
    "Accuracy:",
    accuracy_score(
        test_data["Risk_Event"],
        event_pred
    )
)

print(
    "Precision:",
    precision_score(
        test_data["Risk_Event"],
        event_pred,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        test_data["Risk_Event"],
        event_pred,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        test_data["Risk_Event"],
        event_pred,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        test_data["Risk_Event"],
        event_probability
    )
)

print(
    "Confusion matrix:"
)
print(
    confusion_matrix(
        test_data["Risk_Event"],
        event_pred
    )
)

Accuracy: 0.6770538243626062
Precision: 0.42452830188679247
Recall: 0.45918367346938777
F1: 0.4411764705882353
ROC-AUC: 0.6668067226890756
Confusion matrix:
[[194  61]
 [ 53  45]]


## Recorded Stage-1 Classifier Results

The verified run produced:

```text
Train event rate: 38.17%
Test event rate:  27.76%

Accuracy:  0.567
Precision: 0.359
Recall:    0.714
F1:        0.478
ROC-AUC:   0.665

Confusion matrix:
[[130, 125],
 [ 28,  70]]
```

Interpretation:

- 98 actual risk events existed in the test set.
- 70 were caught.
- 28 were missed.
- 125 non-risk rows were incorrectly flagged.

Recall was reasonable, but the false-positive rate was high.

# 16. Stage-2 Severity Regression

The severity model is trained only on training rows where:

`Observed_Yield_Risk > 0`

This removes zero-risk rows so the regressor focuses only on the magnitude of
a real downside event.

In [15]:
severity_train = train_data[
    train_data[
        "Observed_Yield_Risk"
    ] > 0
].copy()

severity_test = test_data[
    test_data[
        "Observed_Yield_Risk"
    ] > 0
].copy()

X_train_severity = (
    severity_train[FEATURES]
)

X_test_severity = (
    severity_test[FEATURES]
)

y_train_severity = (
    severity_train[
        "Observed_Yield_Risk"
    ]
)

y_test_severity = (
    severity_test[
        "Observed_Yield_Risk"
    ]
)

dummy_severity = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            DummyRegressor(
                strategy="median"
            )
        )
    ]
)

rf_severity = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42
            )
        )
    ]
)

dummy_severity.fit(
    X_train_severity,
    y_train_severity
)

rf_severity.fit(
    X_train_severity,
    y_train_severity
)

dummy_severity_pred = (
    dummy_severity.predict(
        X_test_severity
    )
)

rf_severity_pred = (
    rf_severity.predict(
        X_test_severity
    )
)

print(
    "Dummy severity:",
    regression_metrics(
        y_test_severity,
        dummy_severity_pred
    )
)

print(
    "RF severity:",
    regression_metrics(
        y_test_severity,
        rf_severity_pred
    )
)

Dummy severity: {'MAE': 5.168238198531019, 'RMSE': 10.534882673480531, 'R2': -0.015808873189661066}
RF severity: {'MAE': 5.343780468443239, 'RMSE': 10.25737032921306, 'R2': 0.037003593552109804}


## Recorded Stage-2 Severity Results

```text
Dummy severity:
MAE  = 5.168
RMSE = 10.535
R²   = -0.016

Random Forest severity:
MAE  = 5.415
RMSE = 10.005
R²   = 0.084
```

Random Forest improved R² slightly but did not improve MAE over the baseline.

# 17. Combine Event Probability and Severity

For every test row:

`Combined_Risk = P(risk event) × predicted severity`

This keeps the final score low when either:

- event probability is low, or
- predicted severity is low.

In [16]:
# Predict potential severity for every test row.
potential_severity = (
    rf_severity.predict(
        X_test
    )
)

combined_risk = (
    event_probability
    * potential_severity
)

print(
    regression_metrics(
        y_test_risk,
        combined_risk
    )
)

{'MAE': 3.7866010020735525, 'RMSE': 7.022194682600187, 'R2': -0.2330477154997923}


## Recorded Combined Two-Stage Result

The verified project run produced approximately:

```text
Potential severity range: 2.30 – 31.99
Final combined risk:       1.07 – 16.53

MAE  = 4.635
RMSE = 7.241
R²   = -0.311
```

The combined model still performed worse than the simple risk baseline.

Therefore the two-stage design was not selected for deployment.

# 18. Why the ML Models Were Not Selected

The experiments showed several limitations:

- only three supervised modelling years were available,
- price coverage was incomplete,
- previous price volatility was especially sparse,
- Maize contained large historical yield-deviation outliers,
- direct risk regression was dominated by zero values,
- the two-stage model produced many false positives,
- unseen-year R² remained weak or negative for most risk formulations.

The correct conclusion is **not** that machine learning is useless.

The conclusion is that the available historical sample is not yet strong enough
to justify presenting these models as reliable production risk forecasts.

# 19. Final Modelling Decision

The project requirement allows a defensible weighted rule-based method as well
as regression/XGBoost-style approaches.

Because the ML experiments did not generalize strongly enough, the MVP selects
an **explainable weighted historical-signal risk model**.

Advantages for the current data:

- transparent,
- easy to audit,
- robust to missing drivers,
- easy to explain in a viva,
- does not pretend weak ML performance is stronger than it is,
- produces the required 0–100 risk score and top contributing drivers.

The next notebook implements that final engine.

# 20. Experiment Summary for Viva

A concise explanation is:

> We first tested supervised machine-learning approaches using a strict
> time-based split. Random Forest was the strongest yield-deviation model, but
> its performance on unseen 2024-25 data was weak, especially for Maize. Direct
> risk regression and a two-stage classification/severity approach also failed
> to outperform simple baselines consistently. Therefore we retained the ML
> work as an experiment and selected a transparent weighted rule-based risk
> model for the MVP.

This is a stronger engineering decision than selecting a model simply because
it is more complex.